<a href="https://colab.research.google.com/github/Sumitsharma12321/Deep-Learning-demo/blob/main/Keras_tuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.
Path to dataset files: /kaggle/input/pima-indians-diabetes-database


In [4]:
import os

print(os.listdir(path))

['diabetes.csv']


In [5]:
import pandas as pd
import os

file_path = os.path.join(path, "diabetes.csv")
df = pd.read_csv(file_path)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.corr()["Outcome"]

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [7]:
x = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [9]:
X = scaler.fit_transform(x)

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [12]:
model = Sequential()

model.add(Dense(32, activation="relu", input_dim=8))
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="Adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.fit(batch_size=32, x=X_train, y=y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.5831 - loss: 0.7092 - val_accuracy: 0.6039 - val_loss: 0.6885
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6482 - loss: 0.6613 - val_accuracy: 0.6494 - val_loss: 0.6483
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6792 - loss: 0.6226 - val_accuracy: 0.6948 - val_loss: 0.6129
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7085 - loss: 0.5888 - val_accuracy: 0.7208 - val_loss: 0.5844
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7264 - loss: 0.5608 - val_accuracy: 0.7273 - val_loss: 0.5614
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7345 - loss: 0.5379 - val_accuracy: 0.7338 - val_loss: 0.5441
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7427 - loss: 0.5204 - val_accuracy: 0.7532 - val_loss: 0.5291
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7541 - loss: 0.5060 - val_accuracy: 0.7468 - val_loss

In [14]:
# How to select appropriate optimizer
# number of neuron in layer
# how to select no of layer
# All in one model

In [15]:
import keras_tuner as kt

In [16]:
def build_model(hp):

  model = Sequential()
  model.add(Dense(32, activation="relu", input_dim=8))
  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer=hp.Choice("optimizer", values=["adam", "sgd", "rmsprop","adadelta"]),
                loss="binary_crossentropy",
                metrics=["accuracy"])

  return model

In [17]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [18]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [19]:
best_hp = tuner.get_best_hyperparameters()[0]

print(best_hp.values)

{'optimizer': 'adam'}


In [20]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
optimizer: adam
Score: 0.7792207598686218

Trial 3 summary
Hyperparameters:
optimizer: sgd
Score: 0.7077922224998474

Trial 2 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.6948052048683167

Trial 1 summary
Hyperparameters:
optimizer: adadelta
Score: 0.5909090638160706


In [21]:
# you can also extract your model
model = tuner.get_best_models(num_models=1)[0]
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=10,validation_data=(X_test,y_test))

Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.7655 - loss: 0.4929 - val_accuracy: 0.7922 - val_loss: 0.5272
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7655 - loss: 0.4799 - val_accuracy: 0.7857 - val_loss: 0.5207
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7687 - loss: 0.4721 - val_accuracy: 0.7857 - val_loss: 0.5171
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4656 - val_accuracy: 0.7857 - val_loss: 0.5162
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4620 - val_accuracy: 0.7857 - val_loss: 0.5148
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4580 - val_accuracy: 0.7857 - val_loss: 0.5135
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7590 - loss: 0.4556 - val_accuracy: 0.7857 - val_loss: 0.5123
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4526 - val_accuracy: 0

In [23]:
# what is the right number of neuron in this layer?
# Here, we specify the minimum and maximum number of neurons,
# along with the step size

def build_model(hp):

  model = Sequential()

  units=hp.Int("units", min_value=8, max_value=128, step=8)

  model.add(Dense(units=units, activation="relu", input_dim=8))
  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer="Adam",
                loss="binary_crossentropy",
                metrics=["accuracy"])

  return model


In [24]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir')

In [25]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.6363636255264282

Best val_accuracy So Far: 0.7727272510528564
Total elapsed time: 00h 00m 20s


In [26]:
best_hp = tuner.get_best_hyperparameters()[0]

print(best_hp.values)

{'units': 48}


In [27]:
tuner.results_summary()

Results summary
Results in mydir/untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
units: 48
Score: 0.7727272510528564

Trial 1 summary
Hyperparameters:
units: 96
Score: 0.7467532753944397

Trial 0 summary
Hyperparameters:
units: 120
Score: 0.7402597665786743

Trial 2 summary
Hyperparameters:
units: 32
Score: 0.7207792401313782

Trial 4 summary
Hyperparameters:
units: 8
Score: 0.6363636255264282


In [28]:
model= tuner.get_best_models(num_models=1)[0]
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 48)             │           432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481 (1.88 KB)

 Trainable params: 481 (1.88 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=10,validation_data=(X_test,y_test))

Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.7655 - loss: 0.4983 - val_accuracy: 0.7792 - val_loss: 0.4962
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.4838 - val_accuracy: 0.7857 - val_loss: 0.4897
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7704 - loss: 0.4745 - val_accuracy: 0.7792 - val_loss: 0.4867
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4675 - val_accuracy: 0.7857 - val_loss: 0.4843
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4618 - val_accuracy: 0.7792 - val_loss: 0.4842
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4582 - val_accuracy: 0.7792 - val_loss: 0.4837
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4542 - val_accuracy: 0.7662 - val_loss: 0.4856
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4516 - val_accuracy: 0

here we can also find neuron on every layer

In [30]:
# How to select number of layers?
def build_model(hp):

  model = Sequential()
  model.add(Dense(72, activation="relu", input_dim=8))

  for i in range(hp.Int("num_layers", min_value=1, max_value=10)):

    model.add(Dense(units=hp.Int("units_" + str(i), min_value=32, max_value=512, step=32), activation="relu"))

  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer="Adam",
                loss="binary_crossentropy",
                metrics=["accuracy"])

  return model

In [31]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir',
                        project_name='num_layers')

In [32]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 07s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 35s


In [33]:
best_hp = tuner.get_best_hyperparameters()[0]

print(best_hp.values)

{'num_layers': 4, 'units_0': 192, 'units_1': 416, 'units_2': 192, 'units_3': 128, 'units_4': 448, 'units_5': 224, 'units_6': 64, 'units_7': 416, 'units_8': 512, 'units_9': 32}


In [34]:
model = tuner.get_best_models(num_models=1)[0]
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 72)             │           648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 192)            │        14,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 416)            │        80,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 192)            │        80,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 199,849 (780.66 KB)

 Trainable params: 199,849 (780.66 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
model.fit(X_train,y_train,batch_size=32,epochs=25,initial_epoch=10,validation_data=(X_test,y_test))

Epoch 11/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - accuracy: 0.7850 - loss: 0.4466 - val_accuracy: 0.7532 - val_loss: 0.5505
Epoch 12/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8094 - loss: 0.4110 - val_accuracy: 0.7273 - val_loss: 0.5543
Epoch 13/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8062 - loss: 0.4151 - val_accuracy: 0.7468 - val_loss: 0.5576
Epoch 14/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8127 - loss: 0.3905 - val_accuracy: 0.7727 - val_loss: 0.5409
Epoch 15/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8274 - loss: 0.3803 - val_accuracy: 0.7468 - val_loss: 0.5459
Epoch 16/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8176 - loss: 0.3665 - val_accuracy: 0.7662 - val_loss: 0.5368
Epoch 17/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8453 - loss: 0.3424 - val_accuracy: 0.7597 - val_loss: 0.6273
Epoch 18/25
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8534 - loss: 0.3420 - val_accuracy: 0.7143 - 

In [36]:
def build_model(hp):

  model = Sequential()

  counter = 0

  for i in range(hp.Int("num_layers", min_value=1, max_value=10)):

    if counter == 0:

      model.add(
          Dense(
              units=hp.Int("units" + str(i), min_value=8, max_value=128, step=8),
              activation=hp.Choice("activation" + str(i), values=["relu","tanh", "sigmoid"]),
              input_dim=8
          )
      )
      model.add(
          Dropout(
              hp.Choice("dropout" + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5,0.6,0.7,0.8,0.9])
          )
      )
    else:

      model.add(
          Dense(
              units=hp.Int("units" + str(i), min_value=8, max_value=128, step=8),
              activation=hp.Choice("activation" + str(i), values=["relu","tanh","sigmoid"])
          )
      )
      model.add(
          Dropout(
              hp.Choice("dropout" + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5,0.6,0.7,0.8,0.9])
          )
      )

    counter += 1

  model.add(Dense(1,activation="sigmoid"))

  model.compile(optimizer=hp.Choice("optimizer", values=["adam", "sgd", "rmsprop","adadelta","nadam"]),
                loss="binary_crossentropy",
                metrics=["accuracy"])

  return model

In [37]:
import shutil

shutil.rmtree('mydir', ignore_errors=True)

In [38]:
for hp in tuner.oracle.hyperparameters.space:
    print(hp.name)

num_layers
units_0
units_1
units_2
units_3
units_4
units_5
units_6
units_7
units_8
units_9


In [39]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=20,
                        directory='mydir',
                        project_name='final')

In [40]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 20 Complete [00h 00m 12s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.7142857313156128
Total elapsed time: 00h 04m 01s


In [41]:
tuner.get_best_hyperparameters()[0].values


{'num_layers': 5,
 'units0': 88,
 'activation0': 'tanh',
 'dropout0': 0.3,
 'optimizer': 'adam',
 'units1': 48,
 'activation1': 'tanh',
 'dropout1': 0.3,
 'units2': 112,
 'activation2': 'relu',
 'dropout2': 0.5,
 'units3': 48,
 'activation3': 'sigmoid',
 'dropout3': 0.4,
 'units4': 128,
 'activation4': 'relu',
 'dropout4': 0.7,
 'units5': 56,
 'activation5': 'tanh',
 'dropout5': 0.2,
 'units6': 112,
 'activation6': 'relu',
 'dropout6': 0.6,
 'units7': 80,
 'activation7': 'tanh',
 'dropout7': 0.9,
 'units8': 80,
 'activation8': 'relu',
 'dropout8': 0.8,
 'units9': 24,
 'activation9': 'sigmoid',
 'dropout9': 0.5}

In [42]:
best_hp = tuner.get_best_hyperparameters()[0]

for k, v in best_hp.values.items():
    print(k, ":", v)

num_layers : 5
units0 : 88
activation0 : tanh
dropout0 : 0.3
optimizer : adam
units1 : 48
activation1 : tanh
dropout1 : 0.3
units2 : 112
activation2 : relu
dropout2 : 0.5
units3 : 48
activation3 : sigmoid
dropout3 : 0.4
units4 : 128
activation4 : relu
dropout4 : 0.7
units5 : 56
activation5 : tanh
dropout5 : 0.2
units6 : 112
activation6 : relu
dropout6 : 0.6
units7 : 80
activation7 : tanh
dropout7 : 0.9
units8 : 80
activation8 : relu
dropout8 : 0.8
units9 : 24
activation9 : sigmoid
dropout9 : 0.5


In [43]:
model = tuner.get_best_models(num_models=1)[0]
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 88)             │           792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 88)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 48)             │         4,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 112)            │         5,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 48)             │         5,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,377 (87.41 KB)

 Trainable params: 22,377 (87.41 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
model.fit(X_train,y_train,epochs=100,initial_epoch=10,validation_data=(X_test,y_test))

Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 187ms/step - accuracy: 0.6954 - loss: 0.5843 - val_accuracy: 0.7143 - val_loss: 0.5309
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7215 - loss: 0.5348 - val_accuracy: 0.7078 - val_loss: 0.5221
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7378 - loss: 0.5272 - val_accuracy: 0.7078 - val_loss: 0.5222
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7524 - loss: 0.5185 - val_accuracy: 0.7013 - val_loss: 0.5229
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7427 - loss: 0.5095 - val_accuracy: 0.7143 - val_loss: 0.5232
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7459 - loss: 0.5206 - val_accuracy: 0.7078 - val_loss: 0.5246
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7459 - loss: 0.5062 - val_accuracy: 0.6948 - val_loss: 0.5303
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7476 - loss: 0.4915 - val_accuracy: 

In [45]:
best_hp = tuner.get_best_hyperparameters()[0]

for k, v in best_hp.values.items():
    print(k, ":", v)

num_layers : 5
units0 : 88
activation0 : tanh
dropout0 : 0.3
optimizer : adam
units1 : 48
activation1 : tanh
dropout1 : 0.3
units2 : 112
activation2 : relu
dropout2 : 0.5
units3 : 48
activation3 : sigmoid
dropout3 : 0.4
units4 : 128
activation4 : relu
dropout4 : 0.7
units5 : 56
activation5 : tanh
dropout5 : 0.2
units6 : 112
activation6 : relu
dropout6 : 0.6
units7 : 80
activation7 : tanh
dropout7 : 0.9
units8 : 80
activation8 : relu
dropout8 : 0.8
units9 : 24
activation9 : sigmoid
dropout9 : 0.5
